# CSV Schema Validation
Validates each CSV file against the MySQL schema defined for the football database project.

**Checks performed per table:**
- Required columns present (and no unexpected extra columns)
- NOT NULL columns have no missing values
- Data types are compatible (int, decimal, date, enum, char length)
- Primary key uniqueness
- Enum values are within allowed set
- CHAR(n) length constraints
- Composite primary key uniqueness

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from datetime import datetime

# ── Configure this path to your data directory ──────────────────────────────
DATA_DIR = Path(r"D:\课件\NEU\0CS5200 DB Management\project\project_repo\VuongHColemanTChenT-Project\data")
# ─────────────────────────────────────────────────────────────────────────────

print(f"Data directory: {DATA_DIR}")
print(f"Exists: {DATA_DIR.exists()}")
if DATA_DIR.exists():
    csv_files = list(DATA_DIR.glob("*.csv"))
    print(f"CSV files found: {[f.name for f in csv_files]}")

Data directory: D:\课件\NEU\0CS5200 DB Management\project\project_repo\VuongHColemanTChenT-Project\data
Exists: True
CSV files found: ['club.csv', 'coach.csv', 'country.csv', 'league.csv', 'marketvalue.csv', 'match.csv', 'matchperformance.csv', 'player.csv', 'position.csv', 'seasonperformance.csv', 'stadium.csv', 'transfer.csv', 'user.csv']


In [8]:
# ── Schema definition ────────────────────────────────────────────────────────
# Each table entry:
#   columns        : {col_name: {type, nullable, max_len, enum_vals, ...}}
#   primary_key    : list of column names (composite supported)
#   notes          : optional human-readable notes

SCHEMA = {
    "country": {
        "primary_key": ["country_abbr"],
        "columns": {
            "country_abbr":  {"type": "str",  "nullable": False, "max_len": 3},
            "country_name":  {"type": "str",  "nullable": False, "max_len": 100},
        },
    },
    "league": {
        "primary_key": ["league_id"],
        "composite_unique": [["league_name", "season_name"]],
        "columns": {
            "league_id":    {"type": "int",  "nullable": False},
            "league_name":  {"type": "str",  "nullable": False, "max_len": 100},
            "season_name":  {"type": "str",  "nullable": False, "max_len": 50},
        },
    },
    "stadium": {
        "primary_key": ["stadium_id"],
        "columns": {
            "stadium_id":    {"type": "int", "nullable": False},
            "stadium_name":  {"type": "str", "nullable": False, "max_len": 100},
            "capacity":      {"type": "int", "nullable": True},
            "street_number": {"type": "str", "nullable": True,  "max_len": 20},
            "street_name":   {"type": "str", "nullable": True,  "max_len": 100},
            "city":          {"type": "str", "nullable": True,  "max_len": 100},
            "country":       {"type": "str", "nullable": True,  "max_len": 100},
            "phone_number":  {"type": "str", "nullable": True,  "max_len": 25},
        },
    },
    "club": {
        "primary_key": ["club_id"],
        "columns": {
            "club_id":      {"type": "int", "nullable": False},
            "club_name":    {"type": "str", "nullable": False, "max_len": 100},
            "country_abbr": {"type": "str", "nullable": False, "max_len": 3},
            "stadium_id":   {"type": "int", "nullable": True},
            "coach_id":     {"type": "int", "nullable": True},
        },
    },
    "coach": {
        "primary_key": ["coach_id"],
        "columns": {
            "coach_id":   {"type": "int",  "nullable": False},
            "first_name": {"type": "str",  "nullable": False, "max_len": 50},
            "last_name":  {"type": "str",  "nullable": False, "max_len": 50},
            "dob":        {"type": "date", "nullable": True},
            "nationality":{"type": "str",  "nullable": True,  "max_len": 3},
            "club_id":    {"type": "int",  "nullable": True},
        },
    },
    "position": {
        "primary_key": ["position_id"],
        "columns": {
            "position_id":       {"type": "int", "nullable": False},
            "position_name":     {"type": "str", "nullable": False, "max_len": 50},
            "position_category": {"type": "str", "nullable": True,  "max_len": 50},
        },
    },
    "player": {
        "primary_key": ["player_id"],
        "columns": {
            "player_id":      {"type": "int",     "nullable": False},
            "first_name":     {"type": "str",     "nullable": False, "max_len": 50},
            "last_name":      {"type": "str",     "nullable": False, "max_len": 50},
            "dob":            {"type": "date",    "nullable": True},
            "place_of_birth": {"type": "str",     "nullable": True,  "max_len": 100},
            "height_cm":      {"type": "decimal", "nullable": True},
            "preferred_foot": {"type": "enum",    "nullable": True,
                               "enum_vals": {"Left", "Right", "Both"}},
            "position_id":    {"type": "int",     "nullable": True},
            "country_abbr":   {"type": "str",     "nullable": True,  "max_len": 3},
            "club_id":        {"type": "int",     "nullable": True},
        },
    },
    "marketvalue": {
        "primary_key": ["player_id", "market_value_date"],
        "columns": {
            "player_id":         {"type": "int",     "nullable": False},
            "market_value_date": {"type": "date",    "nullable": False},
            "market_value":      {"type": "decimal", "nullable": False},
        },
    },
    "match": {
        "primary_key": ["match_id"],
        "composite_unique": [["home_team_id", "away_team_id", "match_date"]],
        "columns": {
            "match_id":     {"type": "int",  "nullable": False},
            "home_team_id": {"type": "int",  "nullable": False},
            "away_team_id": {"type": "int",  "nullable": False},
            "league_id": {"type": "int",  "nullable": True},
            "match_date":   {"type": "date", "nullable": False},
            "home_score":   {"type": "int",  "nullable": True},
            "away_score":   {"type": "int",  "nullable": True},
            "home_result":  {"type": "enum", "nullable": True,
                             "enum_vals": {"Win", "Loss", "Draw"}},
            "away_result":  {"type": "enum", "nullable": True,
                             "enum_vals": {"Win", "Loss", "Draw"}},
        },
    },
    "transfer": {
        "primary_key": ["transfer_id"],
        "columns": {
            "transfer_id":   {"type": "int",     "nullable": False},
            "player_id":     {"type": "int",     "nullable": False},
            "old_club_id":   {"type": "int",     "nullable": True},
            "new_club_id":   {"type": "int",     "nullable": False},
            "transfer_date": {"type": "date",    "nullable": False},
            "transfer_fee":  {"type": "decimal", "nullable": True},
        },
    },
    "seasonperformance": {
        "primary_key": ["player_id", "league_id"],
        "columns": {
            "player_id":        {"type": "int", "nullable": False},
            "league_id":        {"type": "int", "nullable": False},
            "appearance_count": {"type": "int", "nullable": True},
            "goal_count":       {"type": "int", "nullable": True},
            "assist_count":     {"type": "int", "nullable": True},
            "play_time":        {"type": "int", "nullable": True},
        },
    },
    "matchperformance": {
        "primary_key": ["match_id", "player_id"],
        "columns": {
            "match_id":           {"type": "int",     "nullable": False},
            "player_id":          {"type": "int",     "nullable": False},
            "play_time":          {"type": "int",     "nullable": True},
            "performance_rating": {"type": "decimal", "nullable": True},
        },
    },
    "user": {
        "primary_key": ["username"],
        "columns": {
            "is_admin": {"type": "bool", "nullable": False},
            "username": {"type": "str",  "nullable": False, "max_len": 20},
            "password": {"type": "str",  "nullable": False, "max_len": 40},
        },
    },
}

print(f"Tables defined in schema: {list(SCHEMA.keys())}")

Tables defined in schema: ['country', 'league', 'stadium', 'club', 'coach', 'position', 'player', 'marketvalue', 'match', 'transfer', 'seasonperformance', 'matchperformance', 'user']


In [4]:
# ── Validation helpers ───────────────────────────────────────────────────────

def is_int_compatible(series: pd.Series) -> pd.Series:
    """Return boolean mask of rows that are NOT valid integers (ignoring NaN)."""
    def bad(v):
        if pd.isna(v):
            return False
        try:
            int(float(v))
            return float(v) != int(float(v))  # reject floats like 1.5
        except (ValueError, TypeError):
            return True
    return series.apply(bad)

def is_decimal_compatible(series: pd.Series) -> pd.Series:
    """Return boolean mask of rows that are NOT valid decimals (ignoring NaN)."""
    def bad(v):
        if pd.isna(v):
            return False
        try:
            float(v)
            return False
        except (ValueError, TypeError):
            return True
    return series.apply(bad)

def is_date_compatible(series: pd.Series) -> pd.Series:
    """Return boolean mask of rows that are NOT valid dates (ignoring NaN)."""
    def bad(v):
        if pd.isna(v):
            return False
        try:
            pd.to_datetime(v)
            return False
        except Exception:
            return True
    return series.apply(bad)

def is_bool_compatible(series: pd.Series) -> pd.Series:
    """Return boolean mask of rows that are NOT valid booleans (ignoring NaN)."""
    VALID = {0, 1, '0', '1', 'true', 'false', True, False}
    def bad(v):
        if pd.isna(v):
            return False
        return str(v).strip().lower() not in {'0','1','true','false'}
    return series.apply(bad)


def validate_table(table_name: str, schema_def: dict, df: pd.DataFrame) -> list:
    """
    Run all checks for one table. Returns a list of issue strings.
    An empty list means the CSV passed all checks.
    """
    issues = []
    schema_cols = schema_def["columns"]
    pk_cols = schema_def["primary_key"]

    # Normalise column names to lowercase for comparison
    df.columns = [c.strip().lower() for c in df.columns]
    csv_cols = set(df.columns)
    expected_cols = set(schema_cols.keys())

    # 1. Missing columns
    missing = expected_cols - csv_cols
    if missing:
        issues.append(f"  [MISSING COLUMNS] {sorted(missing)}")

    # 2. Extra columns
    extra = csv_cols - expected_cols
    if extra:
        issues.append(f"  [EXTRA COLUMNS]   {sorted(extra)}")

    # Work only with columns present in both schema and CSV
    common_cols = expected_cols & csv_cols

    for col in common_cols:
        spec = schema_cols[col]
        s = df[col]

        # 3. NOT NULL check
        if not spec["nullable"]:
            null_count = s.isna().sum()
            if null_count > 0:
                bad_rows = df.index[s.isna()].tolist()[:5]
                issues.append(f"  [NOT NULL] '{col}': {null_count} null(s), e.g. rows {bad_rows}")

        # 4. Type checks
        dtype = spec["type"]

        if dtype == "int":
            bad_mask = is_int_compatible(s)
            if bad_mask.any():
                bad_rows = df.index[bad_mask].tolist()[:5]
                bad_vals = s[bad_mask].tolist()[:5]
                issues.append(f"  [TYPE int] '{col}': {bad_mask.sum()} bad value(s), e.g. {bad_vals} at rows {bad_rows}")

        elif dtype == "decimal":
            bad_mask = is_decimal_compatible(s)
            if bad_mask.any():
                bad_rows = df.index[bad_mask].tolist()[:5]
                bad_vals = s[bad_mask].tolist()[:5]
                issues.append(f"  [TYPE decimal] '{col}': {bad_mask.sum()} bad value(s), e.g. {bad_vals} at rows {bad_rows}")

        elif dtype == "date":
            bad_mask = is_date_compatible(s)
            if bad_mask.any():
                bad_rows = df.index[bad_mask].tolist()[:5]
                bad_vals = s[bad_mask].tolist()[:5]
                issues.append(f"  [TYPE date] '{col}': {bad_mask.sum()} bad value(s), e.g. {bad_vals} at rows {bad_rows}")

        elif dtype == "bool":
            bad_mask = is_bool_compatible(s)
            if bad_mask.any():
                bad_rows = df.index[bad_mask].tolist()[:5]
                bad_vals = s[bad_mask].tolist()[:5]
                issues.append(f"  [TYPE bool] '{col}': {bad_mask.sum()} bad value(s), e.g. {bad_vals} at rows {bad_rows}")

        elif dtype == "enum":
            allowed = spec["enum_vals"]
            bad_mask = s.notna() & ~s.isin(allowed)
            if bad_mask.any():
                bad_vals = s[bad_mask].unique().tolist()[:5]
                issues.append(f"  [ENUM] '{col}': unexpected value(s) {bad_vals} (allowed: {allowed})")

        # 5. Max length check (str columns)
        if "max_len" in spec and dtype == "str":
            max_len = spec["max_len"]
            too_long = s.notna() & (s.astype(str).str.len() > max_len)
            if too_long.any():
                bad_vals = s[too_long].tolist()[:3]
                issues.append(f"  [MAX LEN {max_len}] '{col}': {too_long.sum()} value(s) too long, e.g. {bad_vals}")

    # 6. Primary key uniqueness
    pk_present = [c for c in pk_cols if c in csv_cols]
    if pk_present == pk_cols:  # all PK cols exist
        dups = df.duplicated(subset=pk_cols, keep=False)
        if dups.any():
            dup_vals = df.loc[dups, pk_cols].values.tolist()[:5]
            issues.append(f"  [PK DUPLICATE] on {pk_cols}: {dups.sum()} duplicate row(s), e.g. {dup_vals}")

    # 7. Composite unique key checks
    for uk_cols in schema_def.get("composite_unique", []):
        uk_present = [c for c in uk_cols if c in csv_cols]
        if uk_present == uk_cols:
            dups = df.duplicated(subset=uk_cols, keep=False)
            if dups.any():
                dup_vals = df.loc[dups, uk_cols].values.tolist()[:5]
                issues.append(f"  [UNIQUE KEY DUPLICATE] on {uk_cols}: {dups.sum()} duplicate row(s), e.g. {dup_vals}")

    return issues

print("Validation helpers loaded.")

Validation helpers loaded.


In [9]:
# ── Run validation for all tables ────────────────────────────────────────────

all_results = {}   # table_name -> {"status": ..., "issues": [...], "shape": ...}

for table_name, schema_def in SCHEMA.items():
    csv_path = DATA_DIR / f"{table_name}.csv"

    print(f"\n{'='*60}")
    print(f"Table: {table_name.upper()}   →   {csv_path.name}")
    print('='*60)

    if not csv_path.exists():
        msg = f"  [FILE NOT FOUND] {csv_path}"
        print(msg)
        all_results[table_name] = {"status": "MISSING FILE", "issues": [msg], "shape": None}
        continue

    try:
        df = pd.read_csv(csv_path, dtype=str, keep_default_na=False, na_values=["", "NULL", "null", "None", "none", "NA", "N/A"])
    except Exception as e:
        msg = f"  [READ ERROR] {e}"
        print(msg)
        all_results[table_name] = {"status": "READ ERROR", "issues": [msg], "shape": None}
        continue

    print(f"  Rows: {len(df):,}   Columns: {list(df.columns)}")

    issues = validate_table(table_name, schema_def, df)

    if issues:
        status = "FAIL"
        print(f"  ❌ {len(issues)} issue(s) found:")
        for iss in issues:
            print(iss)
    else:
        status = "PASS"
        print("  ✅ All checks passed!")

    all_results[table_name] = {"status": status, "issues": issues, "shape": df.shape}


Table: COUNTRY   →   country.csv
  Rows: 45   Columns: ['country_abbr', 'country_name']
  ✅ All checks passed!

Table: LEAGUE   →   league.csv
  Rows: 182   Columns: ['league_id', 'season_name', 'league_name']
  ✅ All checks passed!

Table: STADIUM   →   stadium.csv
  Rows: 15   Columns: ['stadium_id', 'stadium_name', 'capacity', 'street_number', 'street_name', 'city', 'country', 'phone_number']
  ✅ All checks passed!

Table: CLUB   →   club.csv
  Rows: 57   Columns: ['club_id', 'club_name', 'country_abbr', 'stadium_id', 'coach_id']
  ✅ All checks passed!

Table: COACH   →   coach.csv
  Rows: 15   Columns: ['coach_id', 'first_name', 'last_name', 'dob', 'nationality', 'club_id']
  ✅ All checks passed!

Table: POSITION   →   position.csv
  Rows: 13   Columns: ['position_id', 'position_name', 'position_category']
  ✅ All checks passed!

Table: PLAYER   →   player.csv
  Rows: 317   Columns: ['player_id', 'first_name', 'last_name', 'dob', 'place_of_birth', 'height_cm', 'preferred_foot', 'p

In [10]:
# ── Summary table ─────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("VALIDATION SUMMARY")
print("="*60)

summary_rows = []
for table, result in all_results.items():
    shape_str = f"{result['shape'][0]:,} rows" if result['shape'] else "—"
    issue_count = len(result['issues'])
    icon = "✅" if result['status'] == "PASS" else "❌"
    summary_rows.append({
        "Table": table,
        "File": f"{table}.csv",
        "Rows": shape_str,
        "Status": f"{icon} {result['status']}",
        "Issues": issue_count,
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

total = len(all_results)
passed = sum(1 for r in all_results.values() if r['status'] == 'PASS')
print(f"\n{passed}/{total} tables passed all checks.")


VALIDATION SUMMARY


,Table,File,Rows,Status,Issues
0,country,country.csv,45 rows,✅ PASS,0
1,league,league.csv,182 rows,✅ PASS,0
2,stadium,stadium.csv,15 rows,✅ PASS,0
3,club,club.csv,57 rows,✅ PASS,0
4,coach,coach.csv,15 rows,✅ PASS,0
5,position,position.csv,13 rows,✅ PASS,0
6,player,player.csv,317 rows,✅ PASS,0
7,marketvalue,marketvalue.csv,"6,815 rows",✅ PASS,0
8,match,match.csv,143 rows,✅ PASS,0
9,transfer,transfer.csv,164 rows,✅ PASS,0



13/13 tables passed all checks.
